# Data Preprocessing

This notebook prepares the dataset for machine learning by separating the features and target variable, splitting the data into training and testing sets, applying feature scaling when necessary, and addressing the severe class imbalance identified during the exploratory data analysis.

## 1. Load Dataset

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/raw/creditcard.csv")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Feature and Target Separation

Machine learning models require the dataset to be divided into input features (`X`) and the target variable (`y`).

The input features contain the information used by the model to make predictions, while the target variable represents the value to be predicted.

In [14]:
x = df.drop("Class", axis=1)
y = df["Class"]

print(f"Shape of x: {x.shape}")
print(f"Shape of y: {y.shape}")

Shape of x: (284807, 30)
Shape of y: (284807,)


**Insights**

- The feature matrix (`X`) contains all transaction attributes except the target variable.
- The target vector (`y`) contains the fraud labels, where `0` represents legitimate transactions and `1` represents fraudulent transactions.

## 3. Train-Test Split

To evaluate the model fairly, the dataset is divided into training and testing subsets.

The training set is used to learn patterns from the data, while the testing set is reserved for evaluating the model on unseen observations.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

print("-----------------------------")

# Verify the class distribution in the original dataset, training set, and test set

print("Original dataset:")
print(y.value_counts(normalize=True))
print("\nTraining set:")
print(y_train.value_counts(normalize=True))
print("\nTest set:")
print(y_test.value_counts(normalize=True))

(227845, 30)
(56962, 30)
-----------------------------
Original dataset:
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

Training set:
Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64

Test set:
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


**Insights**

- The dataset was split into training and testing sets using an 80/20 ratio.
- Stratified sampling was applied to preserve the original class distribution in both subsets.

## 4. Feature Scaling Strategy

Different machine learning algorithms have different preprocessing requirements.

Distance-based and gradient-based algorithms are sensitive to the scale of the input features, whereas tree-based models are generally scale-invariant.

Therefore, the `Amount` feature is standardized to support algorithms that require feature scaling.

In [16]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled["Amount"] = scaler.fit_transform(
    X_train[["Amount"]]
)

X_test_scaled["Amount"] = scaler.transform(
    X_test[["Amount"]]
)

**Insights**

- Only the `Amount` feature was standardized, as the PCA-transformed features are already normalized.
- The scaler was fitted using only the training data to prevent data leakage.
- The same transformation was applied to the testing data using the parameters learned from the training set.

## 5. Class Imbalance Strategy

The exploratory data analysis revealed a severe class imbalance, with fraudulent transactions representing only a small fraction of the dataset.

Although several techniques exist to address this issue, no resampling method is applied during preprocessing. Instead, the original class distribution is preserved so that different imbalance handling strategies can be fairly evaluated during the model training phase.

In [17]:
print("Training set:")
print(y_train.value_counts())

print("\nTesting set:")
print(y_test.value_counts())

print("Training set (%)")
print(y_train.value_counts(normalize=True))

print("\nTesting set (%)")
print(y_test.value_counts(normalize=True))

Training set:
Class
0    227451
1       394
Name: count, dtype: int64

Testing set:
Class
0    56864
1       98
Name: count, dtype: int64
Training set (%)
Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64

Testing set (%)
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


**Insights**

- The severe class imbalance remains present in both the training and testing datasets.
- Stratified sampling successfully preserved the original class distribution after the train-test split.
- Preserving the original distribution allows different imbalance handling techniques to be compared objectively during model training.

## 6. Final Dataset Overview

After completing the preprocessing steps, the dataset is ready for model training.

The following summary confirms the dimensions of the training and testing datasets that will be used throughout the machine learning phase.

In [18]:
print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)

print("----------------------------")

print("Training Labels :", y_train.shape)
print("Testing Labels  :", y_test.shape)

Training Features: (227845, 30)
Testing Features : (56962, 30)
----------------------------
Training Labels : (227845,)
Testing Labels  : (56962,)
